In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -x bin/micromamba ]; then
  curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/chem-mamba
bin/micromamba create -y -p /content/chem-psi4 -c conda-forge --override-channels python=3.12 psi4=1.11 rdkit matplotlib scipy
/content/chem-psi4/bin/python -c 'import psi4, rdkit; print(psi4.__version__, rdkit.__version__)'

固定作者原始甲醇計算的幾何，比對軌域索引並匯出 methanol.fchk。無隨機構形生成。

選擇 Colab 2026.07 的 CPU 執行階段，由上往下執行。安裝格只在目前的雲端工作階段建立獨立環境。

In [ ]:
from pathlib import Path
import os
import subprocess

python = "/content/chem-psi4/bin/python"
def run(script):
    # 子程序只存圖，不沿用 Colab 核心的 inline 顯示後端。
    env = dict(os.environ, MPLBACKEND="Agg")
    result = subprocess.run([python, "-c", script], capture_output=True, text=True, env=env)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()

### 計算

摘要會印在下方。完整的 Psi4 輸出保存在 05-03-frontier-orbitals.log，可以從工作階段的檔案面板開啟。

In [ ]:
run(r'''
import psi4
psi4.core.set_output_file("05-03-frontier-orbitals.log", False)
psi4.set_num_threads(2)
psi4.set_memory("1 GB")
methanol = psi4.geometry("""
0 1
C -0.660110446961  0.259591312943  0.097933867804
O  0.606292553039 -0.291476687057 -0.121678132196
H -0.861264446961  0.336100312943  1.188582867804
H -0.726113446961  1.266898312943 -0.367691132196
H -1.427376446961 -0.397005687057 -0.361404132196
H  1.252273553039  0.329044312943  0.305550867804
units angstrom
symmetry c1
""")
energy, wfn = psi4.energy("B3LYP/6-31G(d)", molecule=methanol, return_wfn=True)
eps = wfn.epsilon_a_subset("AO", "ALL").np
n = wfn.nalpha()
assert n == wfn.nbeta() == 9
homo, lumo = eps[n - 1], eps[n]
print(f"HOMO {homo:.6f} Eh; LUMO {lumo:.6f} Eh; gap {lumo-homo:.6f} Eh")
psi4.set_options({"cubeprop_tasks": ["ORBITALS"], "cubeprop_orbitals": [n, n + 1]})
psi4.cubeprop(wfn)
psi4.fchk(wfn, "methanol.fchk")
''')

### 與原作者的計算比對
圖中為原始 notebook 的四條能階。對照上格採固定座標重新計算的數值。從 Colab 的檔案面板下載 methanol.fchk，可在 Avogadro 開啟密度以外的軌域等值面。

In [ ]:
run(r'''
from pathlib import Path
import matplotlib.pyplot as plt
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figdir.mkdir(parents=True, exist_ok=True)
# 出自作者 psi4_condacolab.ipynb 最後一格甲醇輸出（Psi4 1.10）
levels = [-0.327706, -0.261540, 0.067874, 0.129821]
names = ["8   HOMO−1", "9   HOMO", "10   LUMO", "11   LUMO+1"]
fig, ax = plt.subplots(figsize=(5.7, 3.9))
for i, (value, name) in enumerate(zip(levels, names)):
    ax.hlines(value, 0.4, 1.25, color="#287090" if i < 2 else "#b85924", lw=3)
    ax.text(1.36, value, f"{name}  ({value:+.3f})", va="center")
ax.annotate("", (0.27, levels[2]), (0.27, levels[1]), arrowprops={"arrowstyle":"<->"})
ax.set(xlim=(0.12, 3.15), ylim=(-0.4, 0.2), ylabel="Orbital energy (Eh)", xticks=[], title="Methanol · B3LYP/6-31G(d)")
fig.tight_layout()
fig.savefig(figdir / "05-03-methanol-levels.png", dpi=180)
''')

In [ ]:
from IPython.display import Image, display
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figdir / "05-03-methanol-levels.png")))

### 從本次計算的 cube 網格畫出空間剖面
兩張圖共用通過 C–O 鍵且垂直於 C–O–H 平面的剖面，使用同一色階。

In [ ]:
run(r'''
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates

def read_cube(path):
    lines = path.read_text().splitlines()
    header = np.fromstring(lines[2], sep=" ")
    natom, origin = int(header[0]), header[1:4]
    grid = np.array([np.fromstring(row, sep=" ") for row in lines[3:6]])
    shape = grid[:, 0].astype(int)
    atoms = np.array([np.fromstring(row, sep=" ")[2:5] for row in lines[6:6+natom]])
    volume = np.fromstring(" ".join(lines[6+natom:]), sep=" ").reshape(shape)
    return atoms, origin, grid[:, 1:], volume

def unit(v):
    return v / np.linalg.norm(v)

fig, axes = plt.subplots(1, 2, figsize=(8.1, 4.2), constrained_layout=True)
for ax, number, title in zip(axes, [9, 10], ["HOMO", "LUMO"]):
    path, = Path(".").glob(f"Psi_a_{number}_*.cube")
    atoms, origin, steps, amplitude = read_cube(path)
    carbon, oxygen, hydroxyl_h = atoms[0], atoms[1], atoms[-1]
    u = unit(oxygen - carbon)
    v = unit(np.cross(u, hydroxyl_h - oxygen))
    axis = np.linspace(-4.8, 4.8, 201)
    x, y = np.meshgrid(axis, axis)
    center = (carbon + oxygen) / 2
    plane = center + x[..., None] * u + y[..., None] * v
    indices = (plane - origin) @ np.linalg.inv(steps)
    sliced = map_coordinates(amplitude, indices.transpose(2, 0, 1), order=1)
    contours = ax.contourf(x, y, sliced, levels=np.linspace(-.20, .20, 21),
                           cmap="RdBu_r", extend="both")
    for atom, label in [(carbon, "C"), (oxygen, "O")]:
        ax.text((atom - center) @ u, 0, label, ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.14", fc="white", ec="#777777", lw=0.6))
    ax.set(xlabel="Along C–O (bohr)", title=title, aspect="equal")
axes[0].set_ylabel("Normal to C–O–H plane (bohr)")
fig.colorbar(contours, ax=axes, shrink=0.74, label="Orbital amplitude (a.u.)")
figdir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figdir.mkdir(parents=True, exist_ok=True)
fig.savefig(figdir / "05-03-methanol-orbitals.png", dpi=180)
''')

In [ ]:
display(Image(filename=str(figdir / "05-03-methanol-orbitals.png")))